IMPORTS AND GLOBAL CONFIGS

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from edaBook import numerical_features

LOADING THE DATASET

In [11]:
df = pd.read_csv("dataset/Dataset of Diabetes .csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (1000, 14)


# DATA CLEANING AND STANDARDIZATION

SANITY FIX, EVEN THO I KNOW NO MISSING VALUES AND DUPES

In [12]:
missing_values = df.isnull().sum()

print(missing_values)
print("Total Missing Values:", missing_values.sum())

df = df.drop_duplicates()

ID           0
No_Pation    0
Gender       0
AGE          0
Urea         0
Cr           0
HbA1c        0
Chol         0
TG           0
HDL          0
LDL          0
VLDL         0
BMI          0
CLASS        0
dtype: int64
Total Missing Values: 0


STANDARDIZING COLUMN VALUES

In [13]:
df["CLASS"] = df["CLASS"].astype(str).str.strip().str.upper()

df["Gender"] = df["Gender"].astype(str).str.strip().str.upper()

print(df["CLASS"].value_counts())
print(df["Gender"].value_counts())

CLASS
Y    844
N    103
P     53
Name: count, dtype: int64
Gender
M    565
F    435
Name: count, dtype: int64


REMOVING IDENTIFIER COLUMNS, IRRELEVANT COLUMNS FOR THE MODELING

In [14]:
id_columns = ["ID", "No_Pation"]

df_model = df.drop(columns=id_columns)

#SEPARATING FEATURES AND TARGET

X = df_model.drop(columns=["CLASS"])
y = df_model["CLASS"]

# FEATURE ENGINEERING:

CATEGORIZING AND BINNING BMI AND AGE TO CAPTURE NON-LINEAR RELATIONSHIPS

In [16]:
#BMI BINNING

bmi_bins = [0, 18.5, 24.9, 29.9, np.inf]
bmi_labels = ["Underweight", "Normal", "Overweight", "Obese"]

X["BMI_Category"] = pd.cut(X["BMI"], bins=bmi_bins, labels=bmi_labels, include_lowest=True)

#AGE BINNING

age_bins = [0, 30, 50, 70, np.inf]
age_labels = ["Young", "Middle-aged", "Senior", "Elderly"]

X["Age_Group"] = pd.cut(X["AGE"], bins=age_bins, labels=age_labels, include_lowest=True)

#CHECKING THE NEW BINNED FEATURES

print(X["BMI_Category"].value_counts())
print(X["Age_Group"].value_counts())

BMI_Category
Obese          526
Overweight     284
Normal         190
Underweight      0
Name: count, dtype: int64
Age_Group
Senior         742
Middle-aged    211
Young           27
Elderly         20
Name: count, dtype: int64


TARGET ENCODING

In [17]:
target_mapping = {
    "N": 0,
    "P": 1,
    "Y": 2
}

y_encoded = y.map(target_mapping)

# DATA PREPPING

IMPUTING AND NORMALIZING NUMERICAL (CONTINUOS) FEATURES

In [22]:
numerical_features = X.select_dtypes(include="number").columns

numeric_transform = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ENCODING CATEGORICAL FEATURES

In [23]:
categorical_features = X.select_dtypes(include="object").columns

categorical_transform = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

COLUMN TRANSFORMER

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transform,
            numerical_features
        ),
        (
            "cat",
            categorical_transform,
            categorical_features
        )
    ]
)